# SUPPLEMENTARY PLOTS 

# TABLE OF CONTENTS 
## Correlation Plots 
This script generates plots showing the correlation between parameters from the acute myeloid leukemia analysis using the standard set of covariates and those from the analysis adjusted for the deprivation index. The p-values are Cauchy p-values, combining results across annotation and frequency masks.
- Plot 1: P-valeus from high confidence varinat set 
- Plot 2: Bruden coefficients from high confidence varinat set 
- Plot 3: P-valeus from synonymous varinat set 
- Plot 4: Bruden coefficients from synonymous varinat set 


This script generates plots showing the correlation between parameters from the acute myeloid leukemia analysis using age at last EHR and age at biosample collection. The p-values are Cauchy p-values, combining results across annotation and frequency masks.
- Plot 5: P-valeus from high confidence varinat set 
- Plot 6: Bruden coefficients from high confidence varinat set 


This script generates plots showing the correlation between parameters from the acute myeloid leukemia analysis using age at last EHR and age at diagnosis. For age at diagnosis, cases use age at earliest EHR corresponding to daignosis, while controls use age at last EHR. The p-values are Cauchy p-values, combining results across annotation and frequency masks.
- Plot 7: P-valeus from high confidence varinat set 
- Plot 8: Bruden coefficients from high confidence varinat set 

This script generates plots showing the correlation between parameters from the acute myeloid leukemia analysis using the standard set of covariates with the deprivation index and those from the analysis adjusted for components of the deprivation index. The p-values are Cauchy p-values, combining results across annotation and frequency masks.
- Plot 9: P-valeus from high confidence varinat set 
- Plot 10: Bruden coefficients from high confidence varinat set 

## Transcript Length Plots 

The following plots show gene transcript length and its relationship to overall gene-level significance. Specifically, transcript length is compared against the Cauchy-based negative log p-values to assess whether gene significance is associated with transcript length. Transcript lengths are reported as the log10 of the number of base pairs.
- Plot 11: P-values from the standard covariates model versus transcript length
- Plot 12: P-values from the deprivation index model versus transcript length


In [ ]:
# Imports the 'os' module, which allows interaction with the operating system (e.g., environment variables, file paths).
import os 
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# Retrieves the value of the environment variable 'WORKSPACE_BUCKET'.
my_bucket = os.getenv('WORKSPACE_BUCKET')

# Displays the value of 'my_bucket' (i.e., the workspace bucket URL or path).
my_bucket


# CORRELATION PLOTS 

In [ ]:
# =============================================================================================================
# Plot 2 : Negative log p-value correlation plot 
# =============================================================================================================

fig, ax = plt.subplots(4, 2, figsize=(10, 20), sharex=True, sharey=True)

# ========================================================================================
# Fixed axis limits (same for all panels)
# ========================================================================================

xmin, xmax = 0, 12
ymin, ymax = 0, 12


# ========================================================================================
# Baseline
# ========================================================================================
letters = ["A", "B", "C", "D"]

baseline_files = []
for chrom in range(1, 23):
    file = pd.read_csv(
        f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr{chrom}',
        sep='\t'
    )
    baseline_files.append(file)

baseline_df = pd.concat(baseline_files, axis=0)

baseline = baseline_df[baseline_df['Group'] == 'Cauchy']
baseline = baseline[['Region','max_MAF','Group','Pvalue','MAC']]
baseline['Pvalue'] = - np.log10(baseline['Pvalue'])
baseline.rename(columns={'Pvalue':'baseline','MAC':'baseline_mac'}, inplace=True)

# ========================================================================================
# Comparison models
# ========================================================================================

paths = [
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr{{}}_deprivation_index',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr{{}}_deprivation',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_biosample_age_high_confidence_chr{{}}',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_diagnosis_age_high_confidence_chr{{}}'
]

titles = [
    "Deprivation Index",
    "Deprivation Components",
    "Age at Collection",
    "Age at Diagnosis"
]

# ========================================================================================
# Plotting
# ========================================================================================

for i, base_path in enumerate(paths):

    comparison_files = []

    for chrom in range(1, 23):
        path = base_path.format(chrom)

        try:
            df = pd.read_csv(path, sep='\t')
            comparison_files.append(df)
        except FileNotFoundError:
            print(f"Missing: {path}")
            continue

    file = pd.concat(comparison_files, axis=0)

    comparison = file[file['Group'] == 'Cauchy']
    comparison = comparison[['Region','max_MAF','Group','Pvalue','MAC']]
    comparison['Pvalue'] = - np.log10(comparison['Pvalue'])
    comparison.rename(columns={'Pvalue':'comparison', 'MAC':'comparison_mac'}, inplace=True)


    comparison = pd.merge(
        comparison,
        baseline,
        on=['Region', 'max_MAF', 'Group']
    )

    comparison['color'] = (comparison['comparison'] > comparison['baseline'])

    correlation = comparison[['baseline','comparison']].corr().iloc[0,1]

    # ====================================================================================
    # Plot
    # ====================================================================================

    ax[i,0].scatter(
        comparison['baseline'],
        comparison['comparison'],
        s=10,
        c=comparison['color'].astype(int),
        cmap='viridis',
        alpha=0.7
    )
    
    ax[i,0].text(
        0.02, 0.98,                     
        letters[i],
        transform=ax[i,0].transAxes,
        fontsize=14,
        fontweight='bold',
        va='top',
        ha='left'
    )
  
    ax[i,0].set_xlim(xmin, xmax)
    ax[i,0].set_ylim(ymin, ymax)


    ax[i,0].plot([xmin, xmax], [xmin, xmax], color='red', linestyle='--')

    ax[i,0].xaxis.set_major_locator(MultipleLocator(1))
    ax[i,0].yaxis.set_major_locator(MultipleLocator(1))

    ax[i,0].tick_params(axis='both', labelsize=8)
    ax[i,0].grid(True)

 
    ax[i,0].set_aspect('equal')

    ax[i,0].set_title(f"{titles[i]} (r = {correlation:.3f})", fontsize=14, fontweight='bold')
    ax[i,0].set_ylabel("Comparison −log10(p-value)", fontsize=14, fontweight='bold')
    
    # Make tick labels larger and bold
    for label in ax[i,0].get_xticklabels():
        label.set_fontsize(14)

    for label in ax[i,0].get_yticklabels():
        label.set_fontsize(14)

    
# =============================================================================================================
#
# =============================================================================================================

# ========================================================================================
# Fixed axis limits (same for all panels)
# ========================================================================================

xmin, xmax = 0, 12
ymin, ymax = 0, 12

# ========================================================================================
# Baseline
# ========================================================================================
letters = ["E", "F", "G", "H"]

baseline_files = []
for chrom in range(1, 23):
    file = pd.read_csv(
        f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_synonymous_chr{chrom}',
        sep='\t'
    )
    baseline_files.append(file)

baseline_df = pd.concat(baseline_files, axis=0)

baseline = baseline_df[baseline_df['Group'] == 'Cauchy']
baseline = baseline[['Region','max_MAF','Group','Pvalue','MAC']]
baseline['Pvalue'] = - np.log10(baseline['Pvalue'])
baseline.rename(columns={'Pvalue':'baseline','MAC':'baseline_mac'}, inplace=True)

# ========================================================================================
# Comparison models
# ========================================================================================

paths = [
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_synonymous_chr{{}}_deprivation_index',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_synonymous_chr{{}}_deprivation',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_biosample_age_synonymous_chr{{}}',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_diagnosis_age_synonymous_chr{{}}'
]

titles = [
    "Deprivation Index",
    "Deprivation Components",
    "Age at Collection",
    "Age at Diagnosis"
]

# ========================================================================================
# Plotting
# ========================================================================================

for i, base_path in enumerate(paths):

    comparison_files = []

    for chrom in range(1, 23):
        path = base_path.format(chrom)

        try:
            df = pd.read_csv(path, sep='\t')
            comparison_files.append(df)
        except FileNotFoundError:
            print(f"Missing: {path}")
            continue

    file = pd.concat(comparison_files, axis=0)

    comparison = file[file['Group'] == 'Cauchy']
    comparison = comparison[['Region','max_MAF','Group','Pvalue','MAC']]
    comparison['Pvalue'] = - np.log10(comparison['Pvalue'])
    comparison.rename(columns={'Pvalue':'comparison', 'MAC':'comparison_mac'}, inplace=True)

    comparison = pd.merge(
        comparison,
        baseline,
        on=['Region', 'max_MAF', 'Group']
    )

    comparison['color'] = (comparison['comparison'] > comparison['baseline'])

    correlation = comparison[['baseline','comparison']].corr().iloc[0,1]

    # ====================================================================================
    # Plot
    # ====================================================================================

    ax[i,1].scatter(
        comparison['baseline'],
        comparison['comparison'],
        s=10,
        c=comparison['color'].astype(int),
        cmap='viridis',
        alpha=0.7
    )
    
    ax[i,1].text(
        0.02, 0.98,                     
        letters[i],
        transform=ax[i,1].transAxes,
        fontsize=14,
        fontweight='bold',
        va='top',
        ha='left'
    )

    ax[i,1].set_xlim(xmin, xmax)
    ax[i,1].set_ylim(ymin, ymax)


    ax[i,1].plot([xmin, xmax], [xmin, xmax], color='red', linestyle='--')

    ax[i,1].xaxis.set_major_locator(MultipleLocator(1))
    ax[i,1].yaxis.set_major_locator(MultipleLocator(1))

    ax[i,1].tick_params(axis='both', labelsize=8)
    ax[i,1].grid(True)

 
    ax[i,1].set_aspect('equal')

    ax[i,1].set_title(f"{titles[i]} (r = {correlation:.3f})", fontsize=14, fontweight='bold')
    ax[i,1].set_ylabel("Comparison −log10(p-value)", fontsize=14, fontweight='bold')
    
    # Make tick labels larger and bold
    for label in ax[i,1].get_xticklabels():
        label.set_fontsize(14)

    for label in ax[i,1].get_yticklabels():
        label.set_fontsize(14)
    
# =============================================================================================================
#
# =============================================================================================================

# Shared x-label
ax[-1,1].set_xlabel("Baseline −log10(p-value)", fontsize=14, fontweight='bold')
ax[-1,0].set_xlabel("Baseline −log10(p-value)", fontsize=14, fontweight='bold')
# Layout
plt.tight_layout()

# Save
fig.savefig(
    "aml_neg_log_p_value_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# =============================================================================================================
# Plot 2 : Burden coefficient correlation plot 
# =============================================================================================================

fig, ax = plt.subplots(4, 2, figsize=(10, 20), sharex=True, sharey=True)

# ========================================================================================
# Fixed axis limits (same for all panels)
# ========================================================================================

xmin, xmax = -0.05, 0.55
ymin, ymax = -0.05, 0.55


# ========================================================================================
# Baseline
# ========================================================================================
letters = ["A", "B", "C", "D"]

baseline_files = []
for chrom in range(1, 23):
    file = pd.read_csv(
        f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr{chrom}',
        sep='\t'
    )
    baseline_files.append(file)

baseline_df = pd.concat(baseline_files, axis=0)

baseline = baseline_df[baseline_df['Group'] != 'Cauchy']
baseline = baseline[['Region','max_MAF','Group','BETA_Burden','MAC']]
baseline.rename(columns={'BETA_Burden':'baseline','MAC':'baseline_mac'}, inplace=True)

# ========================================================================================
# Comparison models
# ========================================================================================

paths = [
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr{{}}_deprivation_index',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr{{}}_deprivation',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_biosample_age_high_confidence_chr{{}}',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_diagnosis_age_high_confidence_chr{{}}'
]

titles = [
    "Deprivation Index",
    "Deprivation Components",
    "Age at Collection",
    "Age at Diagnosis"
]

# ========================================================================================
# Plotting
# ========================================================================================

for i, base_path in enumerate(paths):

    comparison_files = []

    for chrom in range(1, 23):
        path = base_path.format(chrom)

        try:
            df = pd.read_csv(path, sep='\t')
            comparison_files.append(df)
        except FileNotFoundError:
            print(f"Missing: {path}")
            continue

    file = pd.concat(comparison_files, axis=0)

    comparison = file[file['Group'] != 'Cauchy']
    comparison = comparison[['Region','max_MAF','Group','BETA_Burden','MAC']]
    comparison.rename(columns={'BETA_Burden':'comparison', 'MAC':'comparison_mac'}, inplace=True)

    comparison = pd.merge(
        comparison,
        baseline,
        on=['Region', 'max_MAF', 'Group']
    )

    comparison['color'] = (comparison['comparison'] > comparison['baseline'])

    correlation = comparison[['baseline','comparison']].corr().iloc[0,1]

    # ====================================================================================
    # Plot
    # ====================================================================================

    ax[i,0].scatter(
        comparison['baseline'],
        comparison['comparison'],
        s=10,
        c=comparison['color'].astype(int),
        cmap='viridis',
        alpha=0.7
    )
    
    ax[i,0].text(
        0.02, 0.98,                     
        letters[i],
        transform=ax[i,0].transAxes,
        fontsize=14,
        fontweight='bold',
        va='top',
        ha='left'
    )
  
    ax[i,0].set_xlim(xmin, xmax)
    ax[i,0].set_ylim(ymin, ymax)

    ax[i,0].plot([xmin, xmax], [xmin, xmax], color='red', linestyle='--')

    ax[i,0].xaxis.set_major_locator(MultipleLocator(0.05))
    ax[i,0].yaxis.set_major_locator(MultipleLocator(0.05))

    ax[i,0].tick_params(axis='both', labelsize=8)
    ax[i,0].grid(True)

 
    ax[i,0].set_aspect('equal')

    ax[i,0].set_title(f"{titles[i]} (r = {correlation:.3f})", fontsize=14, fontweight='bold')
    ax[i,0].set_ylabel("Comparison β", fontsize=14, fontweight='bold')
    
    # Make tick labels larger and bold
    for label in ax[i,0].get_xticklabels():
        label.set_fontsize(9)

    for label in ax[i,0].get_yticklabels():
        label.set_fontsize(9)
    
# =============================================================================================================
#
# =============================================================================================================

# ========================================================================================
# Fixed axis limits (same for all panels)
# ========================================================================================

xmin, xmax = -0.05, 0.55
ymin, ymax = -0.05, 0.55


# ========================================================================================
# Baseline
# ========================================================================================
letters = ["E", "F", "G", "H"]

baseline_files = []
for chrom in range(1, 23):
    file = pd.read_csv(
        f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_synonymous_chr{chrom}',
        sep='\t'
    )
    baseline_files.append(file)

baseline_df = pd.concat(baseline_files, axis=0)

baseline = baseline_df[baseline_df['Group'] != 'Cauchy']
baseline = baseline[['Region','max_MAF','Group','BETA_Burden','MAC']]
baseline.rename(columns={'BETA_Burden':'baseline','MAC':'baseline_mac'}, inplace=True)

# ========================================================================================
# Comparison models
# ========================================================================================

paths = [
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_synonymous_chr{{}}_deprivation_index',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_synonymous_chr{{}}_deprivation',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_biosample_age_synonymous_chr{{}}',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_diagnosis_age_synonymous_chr{{}}'
]

titles = [
    "Deprivation Index",
    "Deprivation Components",
    "Age at Collection",
    "Age at Diagnosis"
]

# ========================================================================================
# Plotting
# ========================================================================================

for i, base_path in enumerate(paths):

    comparison_files = []

    for chrom in range(1, 23):
        path = base_path.format(chrom)

        try:
            df = pd.read_csv(path, sep='\t')
            comparison_files.append(df)
        except FileNotFoundError:
            print(f"Missing: {path}")
            continue

    file = pd.concat(comparison_files, axis=0)

    comparison = file[file['Group'] != 'Cauchy']
    comparison = comparison[['Region','max_MAF','Group','BETA_Burden','MAC']]
    comparison.rename(columns={'BETA_Burden':'comparison', 'MAC':'comparison_mac'}, inplace=True)

    comparison = pd.merge(
        comparison,
        baseline,
        on=['Region', 'max_MAF', 'Group']
    )

    comparison['color'] = (comparison['comparison'] > comparison['baseline'])

    correlation = comparison[['baseline','comparison']].corr().iloc[0,1]

    # ====================================================================================
    # Plot
    # ====================================================================================

    ax[i,1].scatter(
        comparison['baseline'],
        comparison['comparison'],
        s=10,
        c=comparison['color'].astype(int),
        cmap='viridis',
        alpha=0.7
    )
    
    ax[i,1].text(
        0.02, 0.98,                     
        letters[i],
        transform=ax[i,1].transAxes,
        fontsize=14,
        fontweight='bold',
        va='top',
        ha='left'
    )

    ax[i,1].set_xlim(xmin, xmax)
    ax[i,1].set_ylim(ymin, ymax)


    ax[i,1].plot([xmin, xmax], [xmin, xmax], color='red', linestyle='--')

    ax[i,1].xaxis.set_major_locator(MultipleLocator(0.05))
    ax[i,1].yaxis.set_major_locator(MultipleLocator(0.05))

    ax[i,1].tick_params(axis='both', labelsize=8)
    ax[i,1].grid(True)

 
    ax[i,1].set_aspect('equal')

    ax[i,1].set_title(f"{titles[i]} (r = {correlation:.3f})", fontsize=14, fontweight='bold')
    ax[i,1].set_ylabel("Comparison β", fontsize=14, fontweight='bold')
    
    # Make tick labels larger and bold
    for label in ax[i,1].get_xticklabels():
        label.set_fontsize(9)

    for label in ax[i,1].get_yticklabels():
        label.set_fontsize(9)
    
# =============================================================================================================
#
# =============================================================================================================

# Shared x-label
ax[-1,1].set_xlabel("Baseline β", fontsize=14, fontweight='bold')
ax[-1,0].set_xlabel("Baseline β", fontsize=14, fontweight='bold')
# Layout
plt.tight_layout()

# Save
fig.savefig(
    "aml_burden_coefficient_comparison.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# TRANSCRIPT LENGTH PLOTS 

In [ ]:
# ========================================================================================
# Download transcript lengths 
# ========================================================================================

import os
import subprocess

if os.path.exists('gencode.v49.annotation.gff3'):
    print("File exists")
else:
    print("File does not exist")
    print("Downloading file")

    subprocess.run([
        "wget",
        "https://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_49/gencode.v49.annotation.gff3.gz"
    ])

    subprocess.run(['gzip', '-d', 'gencode.v49.annotation.gff3.gz'])

In [ ]:
# ========================================================================================
# Functions 
# ========================================================================================

def parse_attributes(attr_string):
    attrs = {}
    for item in attr_string.split(";"):
        if "=" in item:
            key, value = item.split("=", 1)
            attrs[key] = value
    return attrs

# ========================================================================================
# Extract transcript length 
# ========================================================================================

display = False 

gff_file = "gencode.v49.annotation.gff3"

cols = [
    "seqid", "source", "type", "start", "end",
    "score", "strand", "phase", "attributes"
]

chunks = [] 

for i, chunk in enumerate(pd.read_csv(gff_file, sep="\t", comment="#", header=None, names=cols, chunksize=100000)): 
    
    if display: 
        print(f"Loading chunk {i}")
    
    chunk['attr_dict'] = chunk['attributes'].apply(parse_attributes)
    
    chunk["gene_id"] = chunk["attr_dict"].apply(lambda x: x.get("gene_id"))
    chunk["gene_type"] = chunk["attr_dict"].apply(lambda x: x.get("gene_type"))
    chunk["gene_name"] = chunk["attr_dict"].apply(lambda x: x.get("gene_name"))
    chunk["tag"] = chunk["attr_dict"].apply(lambda x: x.get("tag"))
    
    chunk["gene_id"] = chunk["gene_id"].str.split(".").str[0]
    
    chunk = chunk[chunk['type'] == 'transcript']
    chunk = chunk[chunk['gene_type'] == 'protein_coding']
    chunk = chunk[chunk["attributes"].str.contains("MANE_Select", na=False)]
    
    chunk = chunk[["seqid", "source", "type", "start", "end", 
                   "gene_id", "gene_type", "gene_name"]]
    chunk['length'] = (chunk['end'] - chunk['start']) 
    
    chunks.append(chunk)
    
transcript_lengths = pd.concat(chunks, axis=0)

transcript_lengths = transcript_lengths.loc[transcript_lengths.groupby('gene_id')['length'].idxmax()]

In [ ]:
# =============================================================================================================
# Plot 3 : Negative log p-value versus transcript length 
# =============================================================================================================

fig, ax = plt.subplots(5, 1, figsize=(10, 20), sharex=True)

# Base path templates (NO chrom here yet)
paths = [
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr{{}}_deprivation_index',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr{{}}_deprivation',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_phenotype_high_confidence_chr{{}}',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_biosample_age_high_confidence_chr{{}}',
    f'{my_bucket}/data/saige_step02/HC_analysis/EUR_aml_diagnosis_age_high_confidence_chr{{}}'
]

titles = [
    "Deprivation Index",
    "Deprivation Components",
    "Baseline (Age at Last EHR)",
    "Age at Collection",
    "Age at Diagnosis"
]

letters = ["A", "B", "C", "D", "E"]

for i, base_path in enumerate(paths):
    files = []

    for chrom in range(1, 23):
        path = base_path.format(chrom)

        try:
            df = pd.read_csv(path, sep='\t')
            files.append(df)
        except FileNotFoundError:
            print(f"Missing: {path}")
            continue

    file = pd.concat(files, axis=0)

    # Filter
    file = file[file['Group'] == 'Cauchy'][['Region', 'Pvalue']]

    # Correction
    correction = -np.log10(0.05 / len(file))

    # Transform
    file['Pvalue'] = -np.log10(file['Pvalue'])

    # Correct logic
    file['color'] = file['Pvalue'] > correction

    # Merge gene length
    file.rename(columns={'Region': 'gene_id'}, inplace=True)
    file = file.merge(transcript_lengths, on='gene_id')

    file['loglength'] = np.log10(file['length'])

    # ====================================================================================
    # Plot
    # ====================================================================================
    
    ax[i].set_ylim(0, 12)

    ax[i].scatter(
        file['loglength'],
        file['Pvalue'],
        s=14,
        c=np.where(file['color'], 'red', '#3f4a8a')
    )

    ax[i].axhline(y=correction, color='red', linestyle='--')

    ax[i].set_title(titles[i], fontsize=18, fontweight='bold')

    ax[i].xaxis.set_major_locator(MultipleLocator(1))
    ax[i].yaxis.set_major_locator(MultipleLocator(1))

    ax[i].set_axisbelow(True)
    ax[i].grid(True)

    ax[i].set_ylabel("−log10(p-value)", fontsize=16, fontweight='bold')
    
    ax[i].text(
        0.02, 0.98,                     
        letters[i],
        transform=ax[i].transAxes,
        fontsize=14,
        fontweight='bold',
        va='top',
        ha='left'
    )
    
    # Make tick labels larger and bold
    for label in ax[i].get_xticklabels():
        label.set_fontsize(18)

    for label in ax[i].get_yticklabels():
        label.set_fontsize(14)

# =============================================================================================================
#
# =============================================================================================================

ax[-1].set_xlabel("Transcript Length (log10 base pairs)", fontsize=18, fontweight='bold')

plt.tight_layout()
plt.show()

fig.savefig(
    "aml_phenotype_transcript_length.png",
    dpi=300,
    bbox_inches="tight"
)